In [166]:
import numpy as np
import math
import csv
import os
from scipy.special import gammaln
from scipy.optimize import minimize_scalar
from scipy.optimize import fsolve, differential_evolution,minimize

pi = math.pi

In [167]:
def bound_on_Dx(m1, Omega,Tau):
    return m1 * np.exp(-np.power(m1,1-Tau)/ (3 * math.log(m1) ** Omega))

def bound_on_Fx(value, Zeta, Log_space=False):
    if Log_space:
        M1 = m_func(value, Zeta, Log_space)
        return M1 * np.exp(-np.exp(value * (1 - Zeta)) / 4)
    else:
        return m_func(value, Zeta) * np.exp(-np.power(value, 1 - Zeta) / 4)

def bound_on_Tx(m1, K):
    return np.power(m1, -K / 6 + 2)

In [168]:
def zeta_bound(Epsilon):
    return 1 / (1 - Epsilon) * 3 / 4


def epsilon_bound(Zeta):
    return 1 - 3 / (4 * Zeta)

def optimal_epsilon(Zeta):
    return 1 / 2 - 3 / (8 * Zeta)


In [169]:
def binary_entropy(p):
    if p <= 0 or p >= 1:
        return 0.0
    # math.log2 works on floats; if using arrays, use math.log2
    return -(p * math.log2(p) + (1 - p) * math.log2(1 - p))

def m_func(value,Zeta, Log_space=False):
    if Log_space:
        return np.ceil(np.exp(value*Zeta))
    else:
        return np.ceil(np.power(value,Zeta))

def C_epsilon(Epsilon,c0): #2H(˜ε) log 2 + ˜ε log ˜ε − ˜ε
    Tilda_epsilon = Epsilon*c0
    return 2*binary_entropy(Tilda_epsilon)*math.log(2) + Tilda_epsilon*math.log(Tilda_epsilon) - Tilda_epsilon

def exps_bound_of_Ee(value,Epsilon,Delta,Zeta,c0):
    Epsilon_tilda = Epsilon*c0
    L_term = 2*Epsilon+Epsilon_tilda + 4*Zeta*(Delta-Epsilon) -2*Delta
    #linear term is 2ε + ˜ε + 4ζ(δ − ε) -2δ
    constant_term = (2*Epsilon*math.log(3*Delta)+C_epsilon(Epsilon,c0)+2*Delta - 2*Delta*math.log(2*Delta)  + 4*(Delta-Epsilon)*math.log(c0) )

    # print(f"The constant term of Event E is {constant_term:.4e}, while the L_term is {L_term:.4e}")

    exp_term = -3*value/2 - math.log(Delta*(1-1e-30)*(1-Epsilon_tilda)) - math.log(Epsilon_tilda)/2 - 3*math.log(2*pi)/2


    result = constant_term + value*L_term + np.exp(-value)*exp_term #+ 1/(12*Epsilon)*np.exp(-2*value)
    #this result is then multiplied by n, and exponentiated, so value becomes negligible
    print(f"Correct constant term {2*Epsilon*math.log(3*Delta)+C_epsilon(Epsilon,c0)+2*Delta - 2*Delta*math.log(2*Delta)  + 4*(Delta-Epsilon)*math.log(c0)}")
    return result



In [170]:
# Epsilon = 0.25
# Epsilon_prime = Epsilon/(1-Epsilon)
# print(C_epsilon(Epsilon,1),C_epsilon_broken(Epsilon))

In [171]:
def C_epsilon_broken(Epsilon): #H(ε) + H(ε')(1-ε) + ˜ε log ˜ε − ˜ε
    Epsilon_prime = Epsilon/(1-Epsilon)
    return binary_entropy(Epsilon)*math.log(2) + binary_entropy(Epsilon_prime)*(1-Epsilon)*math.log(2) + Epsilon*math.log(Epsilon) - Epsilon



def exps_bound_of_Ee_broken(value,Epsilon,Delta,Zeta,c0):
    Epsilon_prime = Epsilon/(1-Epsilon)
    L_term = 3*Epsilon + 4*Zeta*(Delta-Epsilon) -2*Delta
    #linear term is 3ε + 4ζ(δ − ε) -2δ
    log_alpha = Delta*math.log(3/2)+(Epsilon-Delta)*math.log(3*Delta)
    constant_term = 2*Delta + 2*log_alpha + C_epsilon_broken(Epsilon)

    #C = 2πδc = (2π)^3/2 δε sqrt[(1 − ε)(1 − ε′)].
    C = math.pow(2*pi,3/2)* Delta*Epsilon*math.sqrt((1-Epsilon)*(1-Epsilon_prime))
    exp_term = -3*value/2 -math.log(C)
    print(f"Broken constant term {2*Delta + 2*log_alpha + C_epsilon_broken(Epsilon)}")
    result = constant_term + value*L_term + np.exp(-value)*exp_term #+ 1/(12*Epsilon)*np.exp(-2*value)
    #this result is then multiplied by n, and exponentiated, so value becomes negligible
    return result

In [172]:
def A_condition(value,Zeta,Epsilon,Gamma,Omega,Tau,K,c0,Eta):
    C_k = 2*np.exp(2)*(4*K+1)*(2*K+1) + 2*np.exp(2)
    # C_k = 2*np.exp(2)*(4*K)*(2*K)
    M1 = m_func(value,Zeta,Log_space=True)
    Epsilon_tilda = Epsilon*c0
    log_M = math.log(M1)
    Term1 = Gamma/(C_k*np.power(M1,2*Tau)*np.power(log_M,2*Eta+2*Omega)) * np.power(1-1/log_M**Omega,4*K*log_M**Eta)
    #coupled with e^L
    Term2 = C_epsilon(Epsilon,c0)
    #constant
    Term3 = -(value/2 + math.log(2*math.pi*Epsilon_tilda)/2 + math.log(1-Epsilon_tilda))
    #coupled with e^-L
    Term4 = Epsilon_tilda
    #coupled with L
    return -Term1*np.exp(value) +Term2+Term3*np.exp(-value)+ Term4*value #+ 1/(12*Epsilon)*np.exp(-2*value)

def H_condition(value,Zeta,Gamma,Epsilon,Tau,c0,Constant1=3/(52*np.exp(2))):#constant 1 is the value that arises from lemma 7.2

    M1 = m_func(value,Zeta,Log_space=True)
    Epsilon_tilda=Epsilon*c0
    Term1 = Constant1*Gamma/np.power(M1,2*Tau)
    #coupled with e^L
    Term2 = C_epsilon(Epsilon,c0)
    #constant
    Term3 = -(value/2 + math.log(2*math.pi*Epsilon_tilda)/2+math.log(1-Epsilon_tilda))
    #coupled with e^-L
    Term4 = Epsilon_tilda
    #coupled with L
    return -Term1*np.exp(value) +Term2+Term3*np.exp(-value)+Term4*value #+ 1/(12*Epsilon)*np.exp(-2*value)


In [173]:
def gamma_n(value,Zeta,Delta,Omega,Tau,Log_space=True):
    if Log_space:
        n_value = np.exp(value)
        M1 = m_func(value,Zeta,Log_space=Log_space)
        return Delta**2/4 - Delta/(4*n_value) - 16*np.power(M1,1-2*Tau)/np.power(math.log(M1),2*Omega)

def thm31_req(value,Zeta,Delta, Omega, Tau, bound,Log_space=True):
    if Log_space:
        M1 = m_func(value,Zeta,Log_space=Log_space)
        return bound > 4/M1 + 16*np.power(M1,1-2*Tau)/np.power(math.log(M1),2*Omega) + Delta/(4*np.exp(value))

def lemma42_req(value,Zeta,Epsilon,Log_space=True):
    M1 = m_func(value, Zeta, Log_space)
    if Log_space:
        return 2 + M1  <= Epsilon**2*np.exp(value)

In [174]:
def log_M_condition(value, threshold_for_m, Eta, K): #threshold is NEGATIVE
    if threshold_for_m >= 0:
        return False
    # Using math.log and python's native ** power operator
    term1 = 2
    term2 = (K * (1 - math.log(K))) / (value ** (1 - Eta))
    term3 = (1 + K * Eta * math.log(value)) / value
    return (term1 + term2 - term3) < threshold_for_m

In [175]:
# def least_L_with_params(zeta, delta, gamma, omega, k, tau=1/2, epsilon=0.25, c0=1+1e-30, tol=1e-9):
def least_L_with_params(zeta, delta, omega, k, eta, threshold_M, tau=1/2, epsilon=0.25, c0=1+1e-30, tol=1e-9):

    # Assuming epsilon and zeta_bound are defined
    params = {
        "zeta": zeta,
        "delta": delta,
        # "gamma": gamma,
        "k": k
    }

    # Define requirements as (Current Value, Boolean Condition, Description)
    tau_lower_bound=0.4
    requirements = {
        "Tau": (tau,1>tau>=tau_lower_bound, f"{tau_lower_bound} <= τ < 1" ),
        "Zeta": (zeta, 1 > zeta > 2 / (2 + tau), f"{2 / (2 + tau):.6f} < ζ < 1"),
        "Delta": (delta, 0< delta < epsilon*(1 - c0/(4*zeta-2)), f" δ < {epsilon*epsilon*(1 - (2+c0)/(4*zeta)):.3e}" ),
        # "Gamma": (gamma, 0 < gamma < delta ** 2 / 4, f"0 < γ < {delta**2 / 4:.2e}"),
        "k": (k, k >12 , "k > 12")
    }

    # Check for failures
    failed = [name for name, (val, met, desc) in requirements.items() if not met]

    if failed:
        # print("═══ PARAMETER CHECK FAILED ═══")
        # for name, (val, met, desc) in requirements.items():
        #     status = "✓" if met else "✗"
        #     print(f"{status} {name:6} : {val:<8} | Required: {desc}")

        return 1e12
        # raise ValueError(f"Constraint violation in: {', '.join(failed)}")

    # print("✓ All parameters satisfy requirements.")

    def log_M_feasible(L):
        M = m_func(value=L, Zeta=zeta, Log_space=True)
        log_M = np.log(M)
        return log_M_condition(log_M, threshold_M, eta, k)

    # Define a helper to check all conditions for a given L
    def is_feasible(L):
        # Local evaluation space variables

        M = m_func(value=L, Zeta=zeta, Log_space=True)
        log_M = np.log(M)
        # New Triangle Parameter Condition based on log^\eta(m) instead of k*log(m)
        # Replacing the loose linear Chernoff/McDiarmid threshold bounds

        gamma_opt = gamma_n(value=L, Zeta=zeta, Delta=delta, Omega=omega, Tau=tau, Log_space=True)
        H_exp = H_condition(value=L, Zeta=zeta, Gamma=gamma_opt, Tau=tau, Epsilon=epsilon, c0=c0)

        # Injects the sharper non-linear triangle concentration criteria into Event A
        A_exp = A_condition(value=L, Zeta=zeta, Epsilon=epsilon, Gamma=gamma_opt,
                                      K=k, Tau=tau, Omega=omega, c0=c0, Eta=eta)

        threshold = -1e-60
        Fx_bound = bound_on_Fx(value=L, Zeta=zeta, Log_space=True)
        Dx_bound = bound_on_Dx(M, omega, tau)
        Ee_bound = exps_bound_of_Ee(value=L, Epsilon=epsilon, Delta=delta, Zeta=zeta, c0=c0)
        broken_bound = exps_bound_of_Ee_broken(value=L, Epsilon=epsilon, Delta=delta, Zeta=zeta, c0=c0)
        conds = [
            Ee_bound < threshold,
            H_exp < threshold,
            A_exp < threshold,
            np.exp(L) * delta > M,
            # Explicit non-linear matching check
        ]
        print(Ee_bound,broken_bound)
        # Sum of early error bounds must strictly clear union criteria mass
        # print(f"{1.0-2*np.exp(-1*log_M) - np.exp(A_exp*np.exp(L)) - np.exp(H_exp*np.exp(L)):.2e}")
        return 2 * (Fx_bound + Dx_bound) < 0.90-2*np.exp(threshold_M*log_M) and all(conds),conds

    # --- DYNAMIC RANGE RESOLUTION ---
    # Since it's no longer a uniform net cut, find the window boundaries manually
    low = 100.0
    high = 700.0



    # Initialize boundaries for the binary/logarithmic search
    search_low = low
    search_high = high
    adjusted_high = None

    # A tolerance threshold determines how precise you want the boundary to be.
    # 1e-5 is usually plenty, but you can adjust it based on your precision needs.

    while (search_high - search_low) > 1e-10:
        # Find the midpoint of the current window
        test_high = (search_low + search_high) / 2.0

        if log_M_feasible(test_high):
            # This point is safe! Save it, and try to push higher.
            adjusted_high = test_high
            search_low = test_high
        else:
            # Asymptotic flip hit. We need to look lower.
            search_high = test_high

    if adjusted_high is None:
        return 1e6  # Interval completely broken or fundamentally unfeasible

    # print(f"L can be at most {adjusted_high}")


    # Reset high boundary safely to the upper limits of the verified active zone
    high = adjusted_high
    best_L = high


    feasible,condss = is_feasible(high)
    if not feasible:
        # print(condss)
        return 1e9


    while (high - low) > tol:
        mid = (low + high) / 2
        # print(f"Mid is ={mid}")
        feasible,_ = is_feasible(mid)
        if feasible:
            # print(f"Passes here")
            best_L = mid
            high = mid  # Try to find a smaller L in the lower half
        else:
            low = mid   # Need a larger L, look in the upper half

    # gamma_opt = gamma_n(value=best_L-1, Zeta=zeta, Delta=delta, Omega=omega, Tau=tau, Log_space=True)
    # print(f"bottle neck is exp_A: {not A_condition(value=best_L-1, Zeta=zeta, Epsilon=epsilon, Gamma=gamma_opt,K=k, Tau=tau, Omega=omega, c0=c0, Eta=eta)<-1e-50}")
    # _,conds_before = is_feasible(best_L-0.4)
    # print(conds_before)
    return best_L

In [176]:
# ZETA= 8.00000000000001043610e-01
# DELTA = 1.549369e-02
# # GAMMA= 5.943461e-05
# OMEGA= 1.6
# k_param=1.201872e+01
# least_L_with_params(zeta=ZETA,delta=DELTA,omega=OMEGA,k=k_param,eta=0.78,c0=1+1e-30,threshold_M=-1)


In [177]:
def objective(params):
    # 1. Unpack exactly matching the order of x0
    zeta, delta, omega, k, tau, eta, threshold_m = params

    # 2. Pass EVERYTHING explicitly by name to eliminate positional shifting
    L = least_L_with_params(
        zeta=zeta,
        delta=delta,
        omega=omega,
        k=k,
        eta=eta,                  # explicitly matched
        threshold_M=threshold_m,  # explicitly matched
        tau=tau,                  # explicitly matched
        epsilon=0.25,
        c0=1+1e-30,
        tol=1e-9
    )
    return L

ZETA= 8.03907761502921314190e-01
DELTA = 4.427442882670378e-02
OMEGA= 1.482061342700993
k_param=84.526220856667095
TAU=0.487848572257193
ETA=7.615425473565973e-03
THRESHOLD_M=-0.00612
# Bounds for (zeta, epsilon, gamma, omega, k)
# Using small buffers (1e-6) to avoid exact boundary issues
bounds = [
    (0.75, 0.9999),  # zeta: 0.8 < zeta < 1
    #(0.0001, 0.25),  # epsilon: typically small positive
    (0.0001,0.25), #delta: small positive
    # (1e-12, 0.1),  # gamma: 0 < gamma < epsilon**4 / 4
    (0.5001, 10.0),  # omega: typically > 1
    (1.0001, 50000.0),  # k: k > 12
    (0.45,0.99999), # tau: 0.5=< tau < 1
    (1e-40,1), #eta <= 1
    (-5,-0.00001)#Threshold:M
]


# Initial guess (starting point for the optimizer)
x0 = [ZETA, DELTA, OMEGA, k_param, TAU, ETA, THRESHOLD_M]
# x0 = [ZETA, EPSILON,DELTA, GAMMA, OMEGA, k_param]

# We use COBYLA or Nelder-Mead because your function is likely not differentiable
res = minimize(
    objective,
    x0,
    method='Nelder-Mead',
    bounds=bounds,
    options={'maxiter': 5000, 'disp': True}
)

if res.success:
    # optimized_zeta, optimized_delta, optimized_gamma, optimized_omega, optimized_k = res.x
    optimized_zeta, optimized_delta, optimized_omega, optimized_k,optimized_tau,optimized_eta,best_threshold = res.x
    my_result = res.fun
    # optimized_zeta, optimized_epsilon,optimized_delta, optimized_gamma, optimized_omega, optimized_k = res.x

    # Check if it actually found a valid structural constraint index
    if my_result < 1000.0:
        print(f"Minimum L found: {my_result:10f}")
        print(f"Value of n : {np.exp(res.fun):.6e}")
    else:
        print("Optimization 'converged' but only found invalid/penalized regions.")

    print(f"ZETA= {optimized_zeta:.20e}")
    # print(f"EPSILON= {optimized_epsilon:.6e}")
    print(f"DELTA = {optimized_delta:.15e}" )
    # print(f"GAMMA= {optimized_gamma:.15e}")
    print(f"OMEGA= {optimized_omega:.15f}")
    print(f"k_param={optimized_k:.15f}")
    print(f"TAU={optimized_tau:.15f}")
    print(f"ETA={optimized_eta:.15e}")
    print(f"THRESHOLD_M={best_threshold:.5f}")


else:
    print("Optimization failed to converge.")

Correct constant term -0.17806208411661395
Broken constant term -0.26301160251431266
-0.1944591603314061 -0.2794086787291048
Correct constant term -0.17806208411661395
Broken constant term -0.26301160251431266
-0.19058018606128257 -0.2755297044589813
Correct constant term -0.17806208411661395
Broken constant term -0.26301160251431266
-0.19251967319634433 -0.27746919159404304
Correct constant term -0.17806208411661395
Broken constant term -0.26301160251431266
-0.1934894167638752 -0.2784389351615739
Correct constant term -0.17806208411661395
Broken constant term -0.26301160251431266
-0.19397428854764065 -0.27892380694533936
Correct constant term -0.17806208411661395
Broken constant term -0.26301160251431266
-0.19421672443952337 -0.2791662428372221
Correct constant term -0.17806208411661395
Broken constant term -0.26301160251431266
-0.19433794238546476 -0.27928746078316347
Correct constant term -0.17806208411661395
Broken constant term -0.26301160251431266
-0.19439855135843542 -0.27934806

In [178]:
ZETA= 8.03907726270922440293e-01
DELTA = 4.427442911752813e-02
OMEGA= 1.482061363562091
k_param=84.526222046435549
TAU=0.487848567730175
ETA=7.615425580758669e-03
THRESHOLD_M=-0.00612

# print(ZETA > 2/(2+TAU))

SSS = least_L_with_params(
        zeta=ZETA,
        delta=DELTA,
        omega=OMEGA,
        k=k_param,
        tau=TAU,
        eta=ETA,
        threshold_M=THRESHOLD_M
    )
if SSS < 1000:
    print(f"{SSS:.100f}")
else:
    print(f"{SSS:.2e}")


Correct constant term -0.17806207942224306
Broken constant term -0.26301159781994166
-0.1944535867856677 -0.2794031051833663
Correct constant term -0.17806207942224306
Broken constant term -0.26301159781994166
-0.19057592963985726 -0.27552544803755585
Correct constant term -0.17806207942224306
Broken constant term -0.26301159781994166
-0.19251475821276248 -0.2774642766104611
Correct constant term -0.17806207942224306
Broken constant term -0.26301159781994166
-0.1934841724992151 -0.2784336908969137
Correct constant term -0.17806207942224306
Broken constant term -0.26301159781994166
-0.19396887964244142 -0.27891839804014
Correct constant term -0.17806207942224306
Broken constant term -0.26301159781994166
-0.19421123321405456 -0.2791607516117532
Correct constant term -0.17806207942224306
Broken constant term -0.26301159781994166
-0.19433240999986115 -0.2792819283975597
Correct constant term -0.17806207942224306
Broken constant term -0.26301159781994166
-0.19439299839276444 -0.279342516790

In [179]:
ZETA= 8.03907726270922440293e-01
DELTA = 4.427442911752813e-02
OMEGA= 1.482061363562091
k_param=84.526222046435549
TAU=0.487848567730175
ETA=7.615425580758669e-03
THRESHOLD_M=-0.00612
# print(ZETA > 2/(2+TAU))
result1 = least_L_with_params(
        zeta=ZETA,
        delta=DELTA,
        omega=OMEGA,
        k=k_param,
        tau=TAU,
        eta=ETA,
        threshold_M=THRESHOLD_M
    )


file_path = 'params_opt_value.csv'
data_row = [result1,ZETA,DELTA,OMEGA,k_param,TAU,ETA,THRESHOLD_M]
# with open('params_opt_value.csv', "a", newline="") as f:
#     writer = csv.writer(f)
#     writer.writerow(data_row)
#     print(f"New data entry created in :{file_path}")
# print(ZETA > 2/(2+TAU))
# ZETA= 8.03999031283763265776e-01
# DELTA = 4.428224435653951e-02
# OMEGA= 1.482431350114056
# k_param=84.564312314334558
# TAU=0.487659717713236
# ETA=7.615236719605971e-03
# THRESHOLD_M=-0.00612

#best value found until here 189.8000200851068370866414625197649002075195

Correct constant term -0.17806207942224306
Broken constant term -0.26301159781994166
-0.1944535867856677 -0.2794031051833663
Correct constant term -0.17806207942224306
Broken constant term -0.26301159781994166
-0.19057592963985726 -0.27552544803755585
Correct constant term -0.17806207942224306
Broken constant term -0.26301159781994166
-0.19251475821276248 -0.2774642766104611
Correct constant term -0.17806207942224306
Broken constant term -0.26301159781994166
-0.1934841724992151 -0.2784336908969137
Correct constant term -0.17806207942224306
Broken constant term -0.26301159781994166
-0.19396887964244142 -0.27891839804014
Correct constant term -0.17806207942224306
Broken constant term -0.26301159781994166
-0.19421123321405456 -0.2791607516117532
Correct constant term -0.17806207942224306
Broken constant term -0.26301159781994166
-0.19433240999986115 -0.2792819283975597
Correct constant term -0.17806207942224306
Broken constant term -0.26301159781994166
-0.19439299839276444 -0.279342516790